## Step 1 — Setup

Imports, random seed, and constants. Run this cell before anything else.

In [22]:
import numpy as np
import pandas as pd

# Reproducibility: same seed → same synthetic data on every rerun
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Panel dimensions
N_ZIPS = 600
START_DATE = "2023-01-01"
END_DATE = "2023-06-30"  # 181 calendar days inclusive (≈180-day window)

print("Setup complete")
print(f"  seed:       {RANDOM_SEED}")
print(f"  n_zips:     {N_ZIPS}")
print(f"  date range: {START_DATE} → {END_DATE}")

Setup complete
  seed:       42
  n_zips:     600
  date range: 2023-01-01 → 2023-06-30


## Step 2 — Base panel generation

1. Build ~600 synthetic 5-digit NJ ZIP codes  
2. Build a daily date range (Jan 1 → Jun 30, 2023)  
3. Create the ZIP × Date panel with random `daily_cases`

In [23]:
# --- 2a. Synthetic NJ ZIP codes (5-digit strings) ---
# Sample without replacement from plausible NJ ZIP ranges (070xx–089xx)
candidates = [f"{z:05d}" for z in range(7001, 8999)]
zip_codes = sorted(rng.choice(candidates, size=N_ZIPS, replace=False))

print(f"ZIPs generated: {len(zip_codes)}")
print(f"Examples:      {zip_codes[:5]}")

ZIPs generated: 600
Examples:      [np.str_('07010'), np.str_('07012'), np.str_('07013'), np.str_('07014'), np.str_('07017')]


In [24]:
# --- 2b. Daily date range ---
dates = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

print(f"Days:  {len(dates)}")
print(f"First: {dates[0].date()}  |  Last: {dates[-1].date()}")

Days:  181
First: 2023-01-01  |  Last: 2023-06-30


In [25]:
# --- 2c. ZIP × Date panel with random daily_cases ---
panel = pd.MultiIndex.from_product(
    [zip_codes, dates],
    names=["zipcode", "date"],
).to_frame(index=False)

# Non-negative integer case counts (Poisson keeps most days small, with some spikes)
panel["daily_cases"] = rng.poisson(lam=5, size=len(panel))

print("Panel shape:", panel.shape)  # expect ~600 * ~181 = ~108,600 rows
print(panel.head())
print()
print(panel["daily_cases"].describe())

Panel shape: (108600, 3)
  zipcode       date  daily_cases
0   07010 2023-01-01            9
1   07010 2023-01-02            8
2   07010 2023-01-03            6
3   07010 2023-01-04            6
4   07010 2023-01-05            2

count    108600.000000
mean          4.998858
std           2.230483
min           0.000000
25%           3.000000
50%           5.000000
75%           6.000000
max          19.000000
Name: daily_cases, dtype: float64


## Step 3 — Temporal derivatives (local physics)

Sort by ZIP + date, then compute:
- `day_of_week` (0=Mon … 6=Sun)
- `velocity` = day-over-day change in cases (dI/dt)
- `acceleration` = day-over-day change in velocity (d²I/dt²)
- `jerk` = day-over-day change in acceleration (d³I/dt³)

All diffs are computed **within each ZIP** so ZIPs never leak into each other.

In [26]:
# --- 3a. Sort + day of week ---
panel = panel.sort_values(["zipcode", "date"]).reset_index(drop=True)
panel["day_of_week"] = panel["date"].dt.dayofweek  # 0=Monday … 6=Sunday

print("Sorted. day_of_week sample:")
print(panel[["zipcode", "date", "daily_cases", "day_of_week"]].head(10))

Sorted. day_of_week sample:
  zipcode       date  daily_cases  day_of_week
0   07010 2023-01-01            9            6
1   07010 2023-01-02            8            0
2   07010 2023-01-03            6            1
3   07010 2023-01-04            6            2
4   07010 2023-01-05            2            3
5   07010 2023-01-06            7            4
6   07010 2023-01-07            2            5
7   07010 2023-01-08            3            6
8   07010 2023-01-09            7            0
9   07010 2023-01-10            4            1


In [27]:
# --- 3b. Velocity, acceleration, jerk (per ZIP) ---
g = panel.groupby("zipcode", sort=False)["daily_cases"]

panel["velocity"] = g.diff()                  # dI/dt
panel["acceleration"] = panel.groupby("zipcode", sort=False)["velocity"].diff()   # d²I/dt²
panel["jerk"] = panel.groupby("zipcode", sort=False)["acceleration"].diff()       # d³I/dt³

# First 1–3 days per ZIP are NaN by construction (not enough history for diffs)
print("Panel shape:", panel.shape)
print()
print(panel[["daily_cases", "velocity", "acceleration", "jerk"]].describe())
print()
print("Null counts (expected early in each ZIP series):")
print(panel[["velocity", "acceleration", "jerk"]].isna().sum())

print(panel.head(15))

Panel shape: (108600, 7)

         daily_cases       velocity   acceleration           jerk
count  108600.000000  108000.000000  107400.000000  106800.000000
mean        4.998858       0.000259      -0.000950       0.001685
std         2.230483       3.155242       5.468946       9.992146
min         0.000000     -16.000000     -28.000000     -47.000000
25%         3.000000      -2.000000      -4.000000      -7.000000
50%         5.000000       0.000000       0.000000       0.000000
75%         6.000000       2.000000       4.000000       7.000000
max        19.000000      15.000000      24.000000      46.000000

Null counts (expected early in each ZIP series):
velocity         600
acceleration    1200
jerk            1800
dtype: int64
   zipcode       date  daily_cases  day_of_week  velocity  acceleration  jerk
0    07010 2023-01-01            9            6       NaN           NaN   NaN
1    07010 2023-01-02            8            0      -1.0           NaN   NaN
2    07010 2023-01-0

## Step 4 — Spatial mobility risk (LODES)

Uses **real LODES-style OD columns**, loaded from a file you can swap later.

**Design**
- Config cell: path / year / column names (only thing you change for new data)
- Loader: always returns the same schema: `origin_zip`, `destination_zip`, `S000`
- Risk step: OD (static) × origin `daily_cases` (daily) → `weighted_infected_inflow` by destination × date
- Merge onto panel with keys: `zipcode` = destination, `date` = date

**LODES reminder**
- Residence / origin = home (`h_geocode` in raw LODES)
- Workplace / destination = work (`w_geocode` in raw LODES)
- `S000` = total jobs on that OD link


In [28]:
# --- 4a. LODES config (swap this block when data updates) ---
from pathlib import Path

LODES_CFG = {
    # Zip-level OD extract (easy to replace with another year/file)
    "path": Path("Data/LEHD/nj_zip_commuter_flows_2021.csv"),
    "year": 2021,
    "level": "zip",  # "zip" now; later "block" for raw LODES od_main
    # Map file columns -> standard names used by the rest of the pipeline
    # Raw LODES od_main would be: origin=h_geocode, destination=w_geocode, jobs=S000
    "columns": {
        "origin": "origin_zip",
        "destination": "destination_zip",
        "jobs": "S000",
    },
}

print("LODES config")
for k, v in LODES_CFG.items():
    print(f"  {k}: {v}")


LODES config
  path: Data/LEHD/nj_zip_commuter_flows_2021.csv
  year: 2021
  level: zip
  columns: {'origin': 'origin_zip', 'destination': 'destination_zip', 'jobs': 'S000'}


In [29]:
# --- 4b. Loader -> standard OD schema ---
def load_lodes_od(cfg: dict) -> pd.DataFrame:
    """Load OD flows and normalize to origin_zip, destination_zip, S000.

    Swap cfg["path"] / cfg["columns"] for new files without changing later steps.
    """
    path = cfg["path"]
    col = cfg["columns"]

    # pandas reads .csv and .csv.gz the same way
    raw = pd.read_csv(path)

    needed = [col["origin"], col["destination"], col["jobs"]]
    missing = [c for c in needed if c not in raw.columns]
    if missing:
        raise ValueError(f"{path} missing columns {missing}. Found: {list(raw.columns)}")

    od = raw[needed].rename(
        columns={
            col["origin"]: "origin_zip",
            col["destination"]: "destination_zip",
            col["jobs"]: "S000",
        }
    ).copy()

    # Standardize join keys as 5-digit strings (matches panel["zipcode"])
    od["origin_zip"] = od["origin_zip"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    od["destination_zip"] = (
        od["destination_zip"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    )
    od["S000"] = pd.to_numeric(od["S000"], errors="coerce").fillna(0).astype(int)

    # One row per OD pair (safe if a future file has duplicates)
    od = od.groupby(["origin_zip", "destination_zip"], as_index=False)["S000"].sum()
    od.attrs["lodes_year"] = cfg["year"]
    od.attrs["lodes_level"] = cfg["level"]
    return od


od_flows = load_lodes_od(LODES_CFG)

print("OD shape:", od_flows.shape)
print("LODES year/level:", od_flows.attrs.get("lodes_year"), od_flows.attrs.get("lodes_level"))
print(od_flows.head())
print()
print(od_flows["S000"].describe())


OD shape: (181994, 3)
LODES year/level: 2021 zip
  origin_zip destination_zip  S000
0      07001           07001   250
1      07001           07002    27
2      07001           07003    17
3      07001           07004    10
4      07001           07005     2

count    181994.000000
mean         18.900755
std          87.359831
min           1.000000
25%           1.000000
50%           4.000000
75%          12.000000
max       16303.000000
Name: S000, dtype: float64


In [30]:
# --- 4c. weighted_infected_inflow (destination x date), then merge to panel ---
def compute_weighted_infected_inflow(panel: pd.DataFrame, od: pd.DataFrame) -> pd.DataFrame:
    """Static OD jobs x origin daily cases, summed to each workplace ZIP-day.

    Join keys back to panel:
      panel.zipcode  <->  destination_zip
      panel.date     <->  date
    """
    origin_cases = panel[["zipcode", "date", "daily_cases"]].rename(
        columns={"zipcode": "origin_zip", "daily_cases": "origin_daily_cases"}
    )

    # Many-to-many on origin_zip: each OD link gets every date for that origin
    linked = od.merge(origin_cases, on="origin_zip", how="inner")
    linked["flow_risk"] = linked["S000"] * linked["origin_daily_cases"]

    inflow = (
        linked.groupby(["destination_zip", "date"], as_index=False)["flow_risk"]
        .sum()
        .rename(
            columns={
                "destination_zip": "zipcode",
                "flow_risk": "weighted_infected_inflow",
            }
        )
    )
    return inflow


inflow = compute_weighted_infected_inflow(panel, od_flows)

panel = panel.merge(inflow, on=["zipcode", "date"], how="left")
panel["weighted_infected_inflow"] = panel["weighted_infected_inflow"].fillna(0.0)

n_panel_zips = panel["zipcode"].nunique()
n_with_inflow = panel.loc[panel["weighted_infected_inflow"] > 0, "zipcode"].nunique()

print("Inflow table shape:", inflow.shape)
print("Panel shape after merge:", panel.shape)
print(f"ZIPs in panel with any inflow: {n_with_inflow} / {n_panel_zips}")
print("(Low overlap is expected while panel ZIPs are synthetic; swap Step 2 ZIPs to real LODES ZIPs later.)")
print()
print(panel[["zipcode", "date", "daily_cases", "weighted_infected_inflow"]].head(10))
print()
print(panel["weighted_infected_inflow"].describe())


Inflow table shape: (106247, 3)
Panel shape after merge: (108600, 8)
ZIPs in panel with any inflow: 180 / 600
(Low overlap is expected while panel ZIPs are synthetic; swap Step 2 ZIPs to real LODES ZIPs later.)

  zipcode       date  daily_cases  weighted_infected_inflow
0   07010 2023-01-01            9                    7363.0
1   07010 2023-01-02            8                    6818.0
2   07010 2023-01-03            6                    5859.0
3   07010 2023-01-04            6                    5135.0
4   07010 2023-01-05            2                    3186.0
5   07010 2023-01-06            7                    5929.0
6   07010 2023-01-07            2                    3276.0
7   07010 2023-01-08            3                    4100.0
8   07010 2023-01-09            7                    6756.0
9   07010 2023-01-10            4                    4661.0

count    108600.000000
mean       2537.437145
std        6879.250572
min           0.000000
25%           0.000000
50%         